In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizerFast, BertForSequenceClassification
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from torch.optim import AdamW
from tqdm.auto import tqdm

C:\Users\O.Midiyanto\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------------------------
# 1. Load Data
# -------------------------
X_train = pd.read_csv('./balanced_X_train.csv')
y_train = pd.read_csv('./balanced_Y_train.csv')
X_test  = pd.read_csv('./balanced_X_test.csv')
y_test  = pd.read_csv('./balanced_Y_test.csv')
X_validation  = pd.read_csv('./balanced_X_validation.csv')
y_validation  = pd.read_csv('./balanced_Y_validation.csv')

label_column = "cvssv3_privileges_required"
train_texts = X_train['english_description'].tolist()
train_labels = y_train[label_column].tolist()
test_texts =  X_test['english_description'].tolist()
test_labels = y_test[label_column].tolist()
validation_texts =  X_validation['english_description'].tolist()
validation_labels = y_validation[label_column].tolist()


# Encode labels
le = LabelEncoder()
le.fit(train_labels)
NUM_CLASSES = len(le.classes_)
train_enc_labels = le.transform(train_labels)
test_enc_labels  = le.transform(test_labels)
validation_enc_labels  = le.transform(validation_labels)

# ——————————————
# Save label classes immediately after encoding
import os, pickle
os.makedirs('../labels', exist_ok=True)
with open('../labels/privileges_required_labels.pkl', 'wb') as f:
    pickle.dump(le.classes_, f)
print(f"Saved label classes ({NUM_CLASSES}) to ../labels/privileges_required_labels.pkl")
# and keep `classes` in memory for evaluation:
classes = list(le.classes_)
# ——————————————

Saved label classes (3) to ../labels/privileges_required_labels.pkl


In [3]:
print(y_train['cvssv3_privileges_required'].value_counts())
print(y_validation['cvssv3_privileges_required'].value_counts())
print(y_test['cvssv3_privileges_required'].value_counts())

cvssv3_privileges_required
NONE    20258
HIGH    20258
LOW     20257
Name: count, dtype: int64
cvssv3_privileges_required
LOW     5065
HIGH    5065
NONE    5064
Name: count, dtype: int64
cvssv3_privileges_required
NONE    6331
LOW     6331
HIGH    6330
Name: count, dtype: int64


In [4]:
# -------------------------
# 2. Tokenization
# -------------------------
tokenizer = BertTokenizerFast.from_pretrained('prajjwal1/bert-small')
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings  = tokenizer(test_texts,  truncation=True, padding=True, max_length=128)
validation_encodings  = tokenizer(validation_texts,  truncation=True, padding=True, max_length=128)

# -------------------------
# 3. Dataset
# -------------------------
class CVEDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = CVEDataset(train_encodings, train_enc_labels)
test_ds  = CVEDataset(test_encodings,  test_enc_labels)
val_ds  = CVEDataset(validation_encodings,  validation_enc_labels)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=16)
val_loader  = DataLoader(val_ds,  batch_size=16)

In [5]:
# -------------------------
# 4. Focal Loss
# -------------------------
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, weight: torch.Tensor = None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
    def forward(self, logits, labels):
        ce = F.cross_entropy(logits, labels, weight=self.weight, reduction='none')
        p_t = torch.exp(-ce)
        loss = ((1 - p_t) ** self.gamma * ce).mean()
        return loss


In [6]:
# -------------------------
# 5. Model Init + Freeze All
# -------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BertForSequenceClassification.from_pretrained(
    'prajjwal1/bert-small',
    num_labels=NUM_CLASSES
).to(device)

# Freeze entire BERT at start
for param in model.bert.parameters():
    param.requires_grad = False

# We'll unfreeze one layer per epoch, starting from the top
encoder_layers = model.bert.encoder.layer
num_layers = len(encoder_layers)

# Prepare optimizer (will be re-built each epoch to pick up new params)
def build_optimizer(model, lr=5e-5):
    return AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

# Instantiate loss (no class weights here, but you could add if needed)
criterion = FocalLoss(gamma=2.0)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-small and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
# -------------------------
# 6. Training with Staged Unfreezing + Early Stopping
# -------------------------
best_val_loss = float('inf')
patience, trials = 2, 0
epoch = 0

while True:
    epoch += 1

    # Unfreeze next layer (top-down)
    layer_to_unfreeze = num_layers - epoch
    if layer_to_unfreeze >= 0:
        for p in encoder_layers[layer_to_unfreeze].parameters():
            p.requires_grad = True
        print(f">>> Epoch {epoch}: Unfroze BERT encoder.layer[{layer_to_unfreeze}]")

    optimizer = build_optimizer(model, lr=5e-5)

    # ---- Train one epoch ----
    model.train()
    train_loss, train_correct, train_count = 0.0, 0, 0
    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch}"):
        optimizer.zero_grad()
        inputs = {k: v.to(device) for k,v in batch.items() if k != 'labels'}
        labels = batch['labels'].to(device)
        logits = model(**inputs).logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=1)
        train_loss    += loss.item() * labels.size(0)
        train_correct += (preds == labels).sum().item()
        train_count   += labels.size(0)

    train_loss /= train_count
    train_acc  = train_correct / train_count
    print(f"  → Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")

    # ---- Validate on val_loader ----
    model.eval()
    val_loss, val_correct, val_count = 0.0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            inputs = {k: v.to(device) for k,v in batch.items() if k != 'labels'}
            labels = batch['labels'].to(device)
            logits = model(**inputs).logits
            loss = criterion(logits, labels)

            preds = logits.argmax(dim=1)
            val_loss    += loss.item() * labels.size(0)
            val_correct += (preds == labels).sum().item()
            val_count   += labels.size(0)

    val_loss /= val_count
    val_acc  = val_correct / val_count
    print(f"  → Val   Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

    # ---- Early stopping & checkpointing ----
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        trials = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print("  ** New best model saved")
    else:
        trials += 1
        print(f"  ** No improvement for {trials} epoch{'s' if trials>1 else ''}")
        if trials >= patience:
            print(f">>> Early stopping at epoch {epoch}")
            break

# Load best model checkpoint
model.load_state_dict(torch.load('best_model.pt'))
print(f">>> Loaded best model from epoch {epoch - trials}")


>>> Epoch 1: Unfroze BERT encoder.layer[3]


Training Epoch 1: 100%|██████████| 3799/3799 [01:26<00:00, 43.99it/s]


  → Train Loss: 0.2589, Acc: 0.7138
  → Val   Loss: 0.1907, Acc: 0.7882
  ** New best model saved
>>> Epoch 2: Unfroze BERT encoder.layer[2]


Training Epoch 2: 100%|██████████| 3799/3799 [02:01<00:00, 31.34it/s]


  → Train Loss: 0.1860, Acc: 0.7873
  → Val   Loss: 0.1541, Acc: 0.8157
  ** New best model saved
>>> Epoch 3: Unfroze BERT encoder.layer[1]


...

  → Train Loss: 0.0579, Acc: 0.9232
  → Val   Loss: 0.1053, Acc: 0.8851
  ** New best model saved


Training Epoch 11: 100%|██████████| 3799/3799 [03:13<00:00, 19.62it/s]


  → Train Loss: 0.0515, Acc: 0.9319
  → Val   Loss: 0.1208, Acc: 0.8864
  ** No improvement for 1 epoch


Training Epoch 12: 100%|██████████| 3799/3799 [03:15<00:00, 19.47it/s]


  → Train Loss: 0.0473, Acc: 0.9362
  → Val   Loss: 0.1129, Acc: 0.8904
  ** No improvement for 2 epochs
>>> Early stopping at epoch 12
>>> Loaded best model from epoch 10


In [8]:
# -------------------------
# 7. Evaluation
# -------------------------
model.eval()
all_preds, all_labels = [], []
test_loss, test_count = 0, 0
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
        labels = batch['labels'].to(device)
        logits = model(**inputs).logits
        loss = criterion(logits, labels)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        test_loss += loss.item() * labels.size(0)
        test_count += labels.size(0)

avg_test_loss = test_loss / test_count
acc = accuracy_score(all_labels, all_preds)
bal_acc = balanced_accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average='weighted')
rec = recall_score(all_labels, all_preds, average='weighted')
f1  = f1_score(all_labels, all_preds, average='weighted')
cm  = confusion_matrix(all_labels, all_preds)
report = classification_report(all_labels, all_preds, target_names=classes, digits=4)

print(f"\nTest Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {acc:.4f}")
print(f"Test Balanced Accuracy: {bal_acc:.4f}")
print(f"Weighted Precision:   {prec:.4f}")
print(f"Weighted Recall:      {rec:.4f}")
print(f"Weighted F1:          {f1:.4f}\n")
print("Confusion Matrix:")
print(cm)
print("\nClassification Report:")
print(report)

Evaluating: 100%|██████████| 1187/1187 [00:18<00:00, 65.76it/s]



Test Loss: 0.1087
Test Accuracy: 0.8860
Test Balanced Accuracy: 0.8860
Weighted Precision:   0.8855
Weighted Recall:      0.8860
Weighted F1:          0.8857

Confusion Matrix:
[[6049  176  105]
 [ 323 5297  711]
 [  86  764 5481]]

Classification Report:
              precision    recall  f1-score   support

        HIGH     0.9367    0.9556    0.9460      6330
         LOW     0.8493    0.8367    0.8429      6331
        NONE     0.8704    0.8657    0.8681      6331

    accuracy                         0.8860     18992
   macro avg     0.8855    0.8860    0.8857     18992
weighted avg     0.8855    0.8860    0.8857     18992



In [9]:
# -------------------------
# 8. Save
# -------------------------

out_dir = '../models/privileges-required'
model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)
print(f"Saved model & tokenizer to {out_dir}")

Saved model & tokenizer to ../models/privileges-required


In [10]:
# -------------------------
# 9. Load & Predict Demo
# -------------------------
# Fungsi predict menggunakan model & label map yang tersimpan
def load_and_predict(prompt: str):
    # load label classes
    with open('../labels/privileges_required_labels.pkl', 'rb') as f:
        classes_local = pickle.load(f)
    # load tokenizer & model
    tok = BertTokenizerFast.from_pretrained(out_dir)
    mdl = BertForSequenceClassification.from_pretrained(out_dir).to(device)
    mdl.eval()
    # tokenisasi input
    inputs = tok(
        prompt,
        truncation=True,
        padding='max_length',
        max_length=512,
        return_tensors='pt'
    )
    inputs = {k: v.to(device) for k,v in inputs.items()}
    # inferensi
    with torch.no_grad():
        logits = mdl(**inputs).logits
        probs = F.softmax(logits, dim=1).squeeze().cpu().numpy()
    idx = probs.argmax()
    return classes_local[idx], probs[idx]*100, probs

# Contoh penggunaan
if __name__ == '__main__':
    sample = (
        'Vulnerability in the Oracle VM VirtualBox product of Oracle Virtualization (component: Core). Supported versions that are affected are Prior to 7.0.24 and prior to 7.1.6. Easily exploitable vulnerability allows high privileged attacker with logon to the infrastructure where Oracle VM VirtualBox executes to compromise Oracle VM VirtualBox. While the vulnerability is in Oracle VM VirtualBox, attacks may significantly impact additional products (scope change). Successful attacks of this vulnerability can result in unauthorized creation, deletion or modification access to critical data or all Oracle VM VirtualBox accessible data as well as unauthorized read access to a subset of Oracle VM VirtualBox accessible data and unauthorized ability to cause a partial denial of service (partial DOS) of Oracle VM VirtualBox. CVSS 3.1 Base Score 7.3 (Confidentiality, Integrity and Availability impacts).'
    )
    label, conf, all_probs = load_and_predict(sample)
    with open('../labels/privileges_required_labels.pkl', 'rb') as f:
        classes_local = pickle.load(f)
    print(f"Prompt: {sample}\nPredicted Privileges Required: {label} ({conf:.2f}%)")
    print("\nProbabilities per class:")
    for cls, p in zip(classes_local, all_probs):
        print(f" - {cls:20s}: {p*100:5.2f}%")


Prompt: Vulnerability in the Oracle VM VirtualBox product of Oracle Virtualization (component: Core). Supported versions that are affected are Prior to 7.0.24 and prior to 7.1.6. Easily exploitable vulnerability allows high privileged attacker with logon to the infrastructure where Oracle VM VirtualBox executes to compromise Oracle VM VirtualBox. While the vulnerability is in Oracle VM VirtualBox, attacks may significantly impact additional products (scope change). Successful attacks of this vulnerability can result in unauthorized creation, deletion or modification access to critical data or all Oracle VM VirtualBox accessible data as well as unauthorized read access to a subset of Oracle VM VirtualBox accessible data and unauthorized ability to cause a partial denial of service (partial DOS) of Oracle VM VirtualBox. CVSS 3.1 Base Score 7.3 (Confidentiality, Integrity and Availability impacts).
Predicted Privileges Required: HIGH (95.77%)

Probabilities per class:
 - HIGH             